In [1]:
import os
import tensorflow as tf
import numpy as np


# ============================================================
# 1. SCALED DOT-PRODUCT ATTENTION
# ============================================================

class ScaledDotProductAttention(tf.keras.layers.Layer):
    """
    Scaled Dot-Product Attention:

        Attention(Q, K, V) =
        softmax(QK^T / sqrt(d_k)) V
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, Q, K, V, mask=None):
        # Key dimension
        d_k = tf.cast(tf.shape(K)[-1], tf.float32)

        # QK^T / sqrt(d_k)
        scores = tf.matmul(Q, K, transpose_b=True)
        scores = scores / tf.math.sqrt(d_k)

        # Apply optional mask
        if mask is not None:
            mask = tf.cast(mask, scores.dtype)
            scores += mask * tf.constant(-1e9, dtype=scores.dtype)

        # Softmax
        attention_weights = tf.nn.softmax(scores, axis=-1)

        # Weighted sum of V
        output = tf.matmul(attention_weights, V)

        return output, attention_weights


/Users/mahamatsilebo/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [10]:
import os
import tensorflow as tf
import numpy as np


# ============================================================
# 1. SCALED DOT-PRODUCT ATTENTION
# ============================================================

class ScaledDotProductAttention(tf.keras.layers.Layer):
    """
    Scaled Dot-Product Attention:

        Attention(Q, K, V) =
        softmax(QK^T / sqrt(d_k)) V
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, Q, K, V, mask=None):
        # Key dimension
        d_k = tf.cast(tf.shape(K)[-1], tf.float32)

        # QK^T / sqrt(d_k)
        scores = tf.matmul(Q, K, transpose_b=True)
        scores = scores / tf.math.sqrt(d_k)

        # Apply optional mask
        if mask is not None:
            mask = tf.cast(mask, scores.dtype)
            scores += mask * tf.constant(-1e9, dtype=scores.dtype)

        # Softmax
        attention_weights = tf.nn.softmax(scores, axis=-1)

        # Weighted sum of V
        output = tf.matmul(attention_weights, V)

        return output, attention_weights



In [18]:

# ============================================================
# 2. PROCESS USER QUERY
# ============================================================

def process_query_input(user_query: str, embedding_dim: int = 8):

    """
    Tokenizes the input, creates synthetic embeddings,
    and applies self-attention.
    """

    # Tokenize
    tokens = user_query.strip().split()

    if len(tokens) == 0:
        print("Error: Input query cannot be empty.")
        return

    seq_len = len(tokens)

    # Create vocabulary
    vocab = {}
    for token in tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

    # Convert tokens to IDs
    token_ids = tf.constant(
        [[vocab[token] for token in tokens]],
        dtype=tf.int32
    )

    # Reproducible embeddings
    tf.random.set_seed(42)

    embedding_layer = tf.keras.layers.Embedding(
        input_dim=len(vocab),
        output_dim=embedding_dim
    )

    embeddings = embedding_layer(token_ids)

    # Self-attention:
    # Q = K = V
    Q = embeddings
    K = embeddings
    V = embeddings

    # Attention layer
    attention_layer = ScaledDotProductAttention()

    output, attention_weights = attention_layer(
        Q=Q,
        K=K,
        V=V
    )

    # ========================================================
    # DISPLAY RESULTS
    # ========================================================

    print("\n" + "=" * 70)
    print("             SCALED DOT-PRODUCT SELF-ATTENTION")
    print("=" * 70)

    print(f"Input Query           : {user_query}")
    print(f"Extracted Tokens      : {tokens}")
    print(f"Number of Tokens      : {seq_len}")
    print(f"Embedding Dimension   : {embedding_dim}")

    print(f"\nInput Embedding Shape : {embeddings.shape}")
    print(f"Attention Output Shape: {output.shape}")
    print(f"Attention Matrix Shape: {attention_weights.shape}")

    # ========================================================
    # ATTENTION MATRIX
    # ========================================================

    print("\n" + "-" * 70)
    print("Attention Weights Matrix")
    print("-" * 70)

    weights_matrix = attention_weights[0].numpy()

    # Header
    print(f"{'Q / K':<15}", end="")

    for token in tokens:
        print(f"{token[:10]:>12}", end="")

    print()

    print("-" * (15 + 12 * seq_len))

    # Rows
    for i, query_token in enumerate(tokens):

        print(f"{query_token[:12]:<15}", end="")

        for j in range(seq_len):
            print(f"{weights_matrix[i][j]:>12.4f}", end="")

        print()

    # ========================================================
    # CHECK ROW SUMS
    # ========================================================

    print("\n" + "-" * 70)
    print("Attention Weight Row Sums")
    print("-" * 70)

    row_sums = weights_matrix.sum(axis=-1)

    for i, token in enumerate(tokens):
        print(f"{token:<15}: {row_sums[i]:.4f}")

    # ========================================================
    # OUTPUT TENSOR
    # ========================================================

    print("\n" + "-" * 70)
    print("Contextual Output Tensor")
    print("-" * 70)

    print(output.numpy())

    return {
        "tokens": tokens,
        "embeddings": embeddings,
        "attention_weights": attention_weights,
        "output": output
    }



In [13]:

# ============================================================
# 3. GEMINI API
# ============================================================

def ask_gemini():

    try:
        from google import genai

    except ImportError:
        print("\nGemini SDK is not installed.")
        print("Install it using:")
        print("pip install google-genai")
        return

    # Get API key from environment variable
    api_key = os.environ.get("GEMINI_API_KEY")

    if not api_key:
        print("\nGemini API key not found.")
        print("Set the GEMINI_API_KEY environment variable first.")
        return

    # Create Gemini client
    client = genai.Client(api_key=api_key)

    # Generate response
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="Explain how embeddings work in one simple sentence."
    )

    print("\n" + "=" * 70)
    print("                       GEMINI RESPONSE")
    print("=" * 70)

    print(response.text)


In [17]:


# ============================================================
# 4. MAIN PROGRAM
# ============================================================

if __name__ == "__main__":

    user_input = input(
        "Enter a query sentence for attention processing: "
    ).strip()

    # Default query
    if not user_input:

        user_input = (
            "Generative AI transforms raw text embeddings"
        )

        print(
            f"\nNo input provided."
            f"\nUsing default query: '{user_input}'"
        )

    # Run attention
    process_query_input(
        user_input,
        embedding_dim=8
    )

    # Run Gemini
    ask_gemini()


             SCALED DOT-PRODUCT SELF-ATTENTION
Input Query           : i am from chad
Extracted Tokens      : ['i', 'am', 'from', 'chad']
Number of Tokens      : 4
Embedding Dimension   : 8

Input Embedding Shape : (1, 4, 8)
Attention Output Shape: (1, 4, 8)
Attention Matrix Shape: (1, 4, 4)

----------------------------------------------------------------------
Attention Weights Matrix
----------------------------------------------------------------------
Q / K                     i          am        from        chad
---------------------------------------------------------------
i                    0.2504      0.2498      0.2499      0.2499
am                   0.2496      0.2505      0.2498      0.2500
from                 0.2497      0.2499      0.2504      0.2501
chad                 0.2497      0.2501      0.2501      0.2502

----------------------------------------------------------------------
Attention Weight Row Sums
--------------------------------------------------------

/Users/mahamatsilebo/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/mahamatsilebo/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)



Gemini API key not found.
Set the GEMINI_API_KEY environment variable first.
